# 08g — Explain (Attention + GNNExplainer, Capacity Revision / GATv2 + pool_anchor)

Post-hoc explainability for `07g`'s trained checkpoints. **Requires `07g`
to have already been run** -- this notebook loads its saved
`{tag}_best_model.pt` / `{tag}_best_model_stats.pt` per scenario, it
does not train anything itself.

**Two techniques, both applied to every explained point:**
1. **Native GATv2 attention-weight extraction** (`explain.extract_attention_weights`)
   -- free (no extra optimization), reads the real per-neighbor attention
   weights the trained model used at each `GATv2Conv` layer.
2. **GNNExplainer-style learned mask** (`explain.run_gnnexplainer`) --
   a small per-point optimization that learns a soft node-importance mask
   (and, for TVG/Unified, an edge-importance mask) that reproduces the
   model's own original prediction as closely as possible while staying
   sparse. Answers "which nodes/edges actually drove THIS prediction,"
   not just "which neighbors got attention at each layer."

Both are implemented directly against this codebase's own
`encoder.forward(data, batch_dict)` signature (`src/explain.py`) rather
than forced through PyTorch Geometric's generic `Explainer` wrapper --
see that module's docstring for why.

**Which points get explained.** For each scenario A-F (G has no GNN
encoder, skipped -- same as `07g` itself), this notebook finds the
SPECIFIC repeat whose weights were kept as `{tag}_best_model.pt`
(`{tag}_best_model_meta.json`'s `repeat` field -- run_scenario_random_repeats
only keeps ONE best-by-val-score repeat's weights per tag, not all 5), then
samples a small number of points **from that repeat's own test split**
(`{tag}_history/repeat{N}_test_predictions.json`) across TP/TN/FP/FN --
so every explained prediction is guaranteed to come from the exact
model whose weights are loaded, not a mismatched repeat/split.

**Normalization.** `07g`'s `train_one_fold` fits per-repeat z-score
stats on that repeat's TRAIN partition only (`ds.fit_normalization`) and
never used to persist them -- `train.py` was extended this session to
save `{tag}_best_model_stats.pt` alongside every `best_model.pt`
specifically so a later notebook like this one can correctly reproduce
the exact input distribution the loaded weights were trained against.
Skipping this step (e.g. explaining raw, unnormalized graphs) would
silently produce meaningless explanations.

**What `pool_anchor` predicts this SHOULD look like.** `07g`'s readout
always concatenates the anchor node's (`ego`/`incident`) own embedding
into the graph vector, by construction (see
`docs/07g_07i_architecture.md` §4). One useful sanity check below: does
GNNExplainer's learned node-importance mask actually rate the anchor
node highly, consistent with the architecture giving it a "free pass"
into every prediction? (Compare against `08i`'s equivalent check, where
`DGCNNReadout` has NO such guarantee.)

GPU recommended for the GNNExplainer optimization loop (small, but runs
once per explained point per branch).

**v2**: adds a feature-level explanation section at the end -- see that section's own markdown intro for what changed and why.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched src/ files until pushed to GitHub.
# Skip this cell once the repo itself is updated -- needs explain.py
# (new this session: extract_attention_weights / run_gnnexplainer /
# explain_scenario_point / aggregate_explanations) and train.py
# (best_model_stats.pt persistence), plus models.py, graph_datasets.py,
# unified_graph.py, baseline_features.py, evaluate.py, plot_history.py.
from google.colab import files
import shutil

print("Upload explain.py, train.py, models.py, graph_datasets.py, unified_graph.py, "
      "baseline_features.py, evaluate.py, plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm seaborn matplotlib

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval_capacity_revision.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_capacity_revision.yaml") as f:
    model_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
COMBINED_PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# MUST match 07g's own dir names exactly -- this notebook only READS
# from CHECKPOINT_DIR/METRICS_DIR, never writes training checkpoints there.
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_capacity_revision"
METRICS_DIR = OUTPUTS_DIR / "metrics_capacity_revision"
assert CHECKPOINT_DIR.exists(), f"{CHECKPOINT_DIR} not found -- run 07g first."
EXPLAIN_DIR = OUTPUTS_DIR / "explain_capacity_revision"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_DEPTH = model_cfg.get("head_depth", "mlp2")

# SET THIS YOURSELF -- how many points per confusion-matrix category
# (TP/TN/FP/FN) to explain, per scenario. No default is assumed here on
# purpose: each explained point costs one GNNExplainer optimization run
# (GNNEXPLAINER_EPOCHS steps), so the right number depends on how much
# time/compute you want to spend and how statistically defensible you
# need the resulting importance averages to be (larger N = less noisy
# per-type means, at roughly linear extra cost).
N_PER_CATEGORY = None
assert N_PER_CATEGORY is not None, "Set N_PER_CATEGORY above before running the rest of this notebook."

GNNEXPLAINER_EPOCHS = 100
GNNEXPLAINER_LR = 0.05
EXPLAIN_SEED = 42

print("Device:", device, "| head_depth:", HEAD_DEPTH)
print("Reading checkpoints from:", CHECKPOINT_DIR)
print("Writing explanations to:", EXPLAIN_DIR)
print(f"N_PER_CATEGORY={N_PER_CATEGORY}, gnnexplainer_epochs={GNNEXPLAINER_EPOCHS}")

In [ ]:
import json
import copy
import random
import pandas as pd
import graph_datasets as ds
import models
import explain
import unified_graph as ug

SVG_DIR = COMBINED_PROCESSED_DIR / "svg_graphs"
TVG_DIR = COMBINED_PROCESSED_DIR / "tvg_graphs"
INDEX_PATH = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
index_df = pd.read_parquet(INDEX_PATH)
assert "city" in index_df.columns, (
    f"'{INDEX_PATH}' has no 'city' column -- this notebook needs 05's combined, "
    "multi-city dataset_index.parquet, not a single-city index.")
print(f"Dataset: {len(index_df)} points available for lookup")

_ref_cache_dir = INTERIM_DIR / "osm_cache" / CITIES[0]
with open(_ref_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(_ref_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB_SIZE = len(json.load(f))
print(f"Unified vocab (post-04b): highway={HIGHWAY_VOCAB_SIZE}, building_type={BUILDING_TYPE_VOCAB_SIZE}")

# Must be IDENTICAL to 07g's own svg_kwargs/tvg_kwargs -- these define the
# model architecture that {tag}_best_model.pt's state_dict was saved
# from; any mismatch (wrong hidden_dim, wrong embed dims, ...) will fail
# to load or silently misalign weights.
svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2,
                   cat_embed_dim=model_cfg.get("cat_embed_dim", 4))
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   building_type_vocab=BUILDING_TYPE_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=model_cfg.get("building_type_embed_dim", 32),
                   highway_embed_dim=model_cfg.get("highway_embed_dim", 8))
FUSION_DIM = model_cfg.get("fusion_dim", 256)
HEAD_HIDDEN = model_cfg.get("head_hidden", 256)
HEAD_DROPOUT = model_cfg.get("head_dropout", 0.3)
print("svg_kwargs:", svg_kwargs)
print("tvg_kwargs:", tvg_kwargs)

## Helpers: load a trained scenario's model+stats, pick which points to explain

In [ ]:
def load_scenario_model(scenario, use_ablation=False):
    """Reconstructs the exact architecture build_model() would have
    produced during 07g, loads {tag}_best_model.pt's weights and
    {tag}_best_model_stats.pt's normalization stats. Returns
    (model, stats, best_repeat, tag)."""
    tag = f"{scenario}_{HEAD_DEPTH}" + ("_ablation" if use_ablation else "")
    model_path = CHECKPOINT_DIR / f"{tag}_best_model.pt"
    stats_path = CHECKPOINT_DIR / f"{tag}_best_model_stats.pt"
    meta_path = CHECKPOINT_DIR / f"{tag}_best_model_meta.json"
    assert model_path.exists(), f"{model_path} not found -- run 07g's scenario {scenario} cell first."
    assert stats_path.exists(), (
        f"{stats_path} not found -- this checkpoint predates the best_model_stats.pt "
        f"persistence change; rerun 07g's scenario {scenario} cell with the current train.py.")

    model = models.build_model(scenario, fusion_dim=FUSION_DIM, head_depth=HEAD_DEPTH,
                                head_hidden=HEAD_HIDDEN, head_dropout=HEAD_DROPOUT,
                                use_ablation=use_ablation, svg_kwargs=svg_kwargs, tvg_kwargs=tvg_kwargs)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=False))
    model.eval()
    stats = torch.load(stats_path, map_location="cpu", weights_only=False)
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    best_repeat = meta.get("repeat")
    return model, stats, best_repeat, tag


def pick_points_to_explain(tag, best_repeat, n_per_category=N_PER_CATEGORY, seed=EXPLAIN_SEED):
    """Reads the SPECIFIC repeat's own raw test-predictions file (the
    repeat whose weights are actually loaded, per best_model_meta.json)
    and samples up to n_per_category points per TP/TN/FP/FN category.
    Returns a list of {"point_id", "category"} dicts."""
    if best_repeat is None:
        print(f"  [{tag}] no best_model_meta.json repeat recorded -- skipping point selection.")
        return []
    pred_path = CHECKPOINT_DIR / f"{tag}_history" / f"repeat{best_repeat}_test_predictions.json"
    if not pred_path.exists():
        print(f"  [{tag}] {pred_path} not found -- skipping.")
        return []
    records = json.loads(pred_path.read_text())
    rng = random.Random(seed)
    by_category = {}
    for r in records:
        by_category.setdefault(r["category"], []).append(r)
    picked = []
    for cat, rows in sorted(by_category.items()):
        sample = rng.sample(rows, min(n_per_category, len(rows)))
        picked.extend({"point_id": r["point_id"], "category": cat} for r in sample)
    return picked

## Run both explanation techniques across scenarios A-F -- normal and ablation, SEPARATELY

Two independent passes, never merged: `NORMAL_SCENARIOS` (A-F,
`use_ablation=False`) and `ABLATION_SCENARIOS` (B-F only -- `A` has no
ablation variant, matching `07g`'s own scenario cells). Each pass
produces its own record list, its own tidy DataFrame, and its own set
of output CSVs (`..._normal.csv` / `..._ablation.csv`) -- an ablation
point's importances are never averaged together with a normal point's
under the same scenario key.

In [ ]:
NORMAL_SCENARIOS = ["A", "B", "C", "D", "E", "F"]
ABLATION_SCENARIOS = ["B", "C", "D", "E", "F"]  # A has no ablation variant


def run_explanation_pass(scenarios, use_ablation):
    records = []
    label = "ablation" if use_ablation else "normal"
    for scenario in scenarios:
        print(f"\n=== [{label}] Scenario {scenario} ===")
        model, stats, best_repeat, tag = load_scenario_model(scenario, use_ablation=use_ablation)
        print(f"  loaded {tag} (best repeat={best_repeat})")

        points = pick_points_to_explain(tag, best_repeat, n_per_category=N_PER_CATEGORY)
        print(f"  explaining {len(points)} points: "
              f"{ {c: sum(1 for p in points if p['category']==c) for c in sorted(set(p['category'] for p in points))} }")

        for p in points:
            pid, cat = p["point_id"], p["category"]
            svg_raw = torch.load(SVG_DIR / f"{pid}.pt", weights_only=False)
            tvg_raw = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
            svg_norm, tvg_norm = ds.apply_normalization(copy.deepcopy(svg_raw), copy.deepcopy(tvg_raw), stats)

            try:
                recs = explain.explain_scenario_point(
                    model, scenario, svg_norm, tvg_norm, point_id=pid, category=cat,
                    gnnexplainer_epochs=GNNEXPLAINER_EPOCHS, gnnexplainer_lr=GNNEXPLAINER_LR)
                records.extend(recs)
            except Exception as e:
                print(f"    !! failed to explain {pid} ({cat}): {type(e).__name__}: {e}")
    print(f"\n[{label}] total explained (point, branch) records: {len(records)}")
    return records


normal_records = run_explanation_pass(NORMAL_SCENARIOS, use_ablation=False)
ablation_records = run_explanation_pass(ABLATION_SCENARIOS, use_ablation=True)

## Aggregate into two tidy CSVs -- normal and ablation kept separate

In [ ]:
explain_df_normal = explain.aggregate_explanations(normal_records)
explain_df_ablation = explain.aggregate_explanations(ablation_records)

explain_df_normal.to_csv(EXPLAIN_DIR / "explanations_capacity_revision_normal.csv", index=False)
explain_df_ablation.to_csv(EXPLAIN_DIR / "explanations_capacity_revision_ablation.csv", index=False)

print(f"Saved {len(explain_df_normal)} rows to {EXPLAIN_DIR / 'explanations_capacity_revision_normal.csv'}")
print(f"Saved {len(explain_df_ablation)} rows to {EXPLAIN_DIR / 'explanations_capacity_revision_ablation.csv'}")
display(explain_df_normal.head(10))
display(explain_df_ablation.head(10))

## Sanity check: does GNNExplainer confirm the anchor node's "free pass"? (normal vs. ablation, separately)

`pool_anchor` (this branch's readout) always concatenates the anchor
node's (`ego` for SVG branches, `incident` for TVG branches) own
embedding into the graph vector, by construction -- it can never be
"voted out" the way it could be in `08i`'s DGCNN readout. This cell
checks whether GNNExplainer's learned node-importance mask reflects
that, computed independently for the normal and ablation passes (the
ablation graphs have an extra `peer_incident` node type / `crash_history`
edge, which could plausibly shift how much weight the anchor itself
carries).

In [ ]:
anchor_types_normal = {"A": "ego", "B": "incident", "C_svg": "ego", "C_tvg": "incident",
                       "D_svg": "ego", "D_tvg": "incident", "E_svg": "ego", "E_tvg": "incident",
                       "F": "incident"}  # F's anchor is "incident" per UnifiedEncoder's convention
anchor_types_ablation = {k: v for k, v in anchor_types_normal.items() if k != "A"}


def build_anchor_summary(explain_df, anchor_types):
    gnne_node = explain_df[(explain_df["source"] == "gnnexplainer") & (explain_df["kind"] == "node")]
    rows = []
    for scen, anchor_nt in anchor_types.items():
        sub = gnne_node[gnne_node["scenario"] == scen]
        anchor_rows = sub[sub["type"] == anchor_nt]
        other_rows = sub[sub["type"] != anchor_nt]
        if len(anchor_rows) == 0:
            continue
        rows.append({
            "scenario": scen, "anchor_type": anchor_nt,
            "anchor_mean_importance": anchor_rows["mean_value"].mean(),
            "other_types_mean_importance": other_rows["mean_value"].mean() if len(other_rows) else float("nan"),
            "n_points": anchor_rows["point_id"].nunique(),
        })
    return pd.DataFrame(rows)


anchor_summary_normal_df = build_anchor_summary(explain_df_normal, anchor_types_normal)
anchor_summary_ablation_df = build_anchor_summary(explain_df_ablation, anchor_types_ablation)

anchor_summary_normal_df.to_csv(EXPLAIN_DIR / "anchor_importance_summary_normal.csv", index=False)
anchor_summary_ablation_df.to_csv(EXPLAIN_DIR / "anchor_importance_summary_ablation.csv", index=False)
display(anchor_summary_normal_df)
display(anchor_summary_ablation_df)

In [ ]:
print("08g explainability run complete.")
print(f"[normal]   {explain_df_normal['point_id'].nunique()} unique points across "
      f"{explain_df_normal['scenario'].nunique()} scenario/branch tags.")
print(f"[ablation] {explain_df_ablation['point_id'].nunique()} unique points across "
      f"{explain_df_ablation['scenario'].nunique()} scenario/branch tags.")
print()
print("Normal outputs:   explanations_capacity_revision_normal.csv, anchor_importance_summary_normal.csv,")
print("                  type_importance_summary_normal.csv, type_importance_top5_normal.csv")
print("Ablation outputs: explanations_capacity_revision_ablation.csv, anchor_importance_summary_ablation.csv,")
print("                  type_importance_summary_ablation.csv, type_importance_top5_ablation.csv")

## Per-scheme (scenario) importance summary -- normal and ablation, separately

Folds each pass's tidy per-point table into two report tables that ARE
directly comparable across scenarios/branches:

- **`type_importance_summary_{normal,ablation}.csv`** -- one row per
  `(scenario, kind, type)`, with `gnnexplainer` and `attention` as
  separate mean-importance columns (attention averaged across both
  `GATv2Conv` layers first). Answers "does `C_svg` lean on `signage`
  more than `A` does".
- **`type_importance_top5_{normal,ablation}.csv`** -- for each
  `(scenario, source)`, the top 5 node/edge types by mean importance,
  ranked -- a skimmable summary of what each scheme actually attends
  to / needs.

In [ ]:
type_pivot_normal_df, type_topn_normal_df = explain.build_type_importance_report(explain_df_normal, top_n=5)
type_pivot_ablation_df, type_topn_ablation_df = explain.build_type_importance_report(explain_df_ablation, top_n=5)

type_pivot_normal_df.to_csv(EXPLAIN_DIR / "type_importance_summary_normal.csv", index=False)
type_topn_normal_df.to_csv(EXPLAIN_DIR / "type_importance_top5_normal.csv", index=False)
type_pivot_ablation_df.to_csv(EXPLAIN_DIR / "type_importance_summary_ablation.csv", index=False)
type_topn_ablation_df.to_csv(EXPLAIN_DIR / "type_importance_top5_ablation.csv", index=False)

print(f"[normal]   saved {len(type_pivot_normal_df)} summary rows, {len(type_topn_normal_df)} top-5 rows")
print(f"[ablation] saved {len(type_pivot_ablation_df)} summary rows, {len(type_topn_ablation_df)} top-5 rows")
display(type_topn_normal_df)
display(type_topn_ablation_df)

## Figures: top-5 importance per scenario, seaborn, L-shaped (despined) axes

Plots `gnnexplainer` importance only, not `attention` -- raw GATv2
attention weights were found (see per-point QC earlier this session) to
be dominated by node-degree ceiling effects (a node with 2 neighbors
can only ever attend `0.5`/`0.5`, regardless of what the model actually
learned), not a real importance signal, so plotting them alongside
GNNExplainer would visually overstate their reliability. Every axes
object below keeps only its left + bottom spines (`sns.despine()`) --
the "L-shaped" academic-figure convention -- on a `whitegrid`-free,
muted-palette background.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("ticks")
sns.set_context("paper")
PALETTE = sns.color_palette("muted")
FIG_DIR = EXPLAIN_DIR


def plot_top_importance_grid(topn_df, title, filename, source="gnnexplainer"):
    """One horizontal bar subplot per scenario, top-5 (scenario, source)
    rows from build_type_importance_report's topn_df. L-shaped axes
    (sns.despine: only left+bottom spines kept)."""
    sub = topn_df[topn_df["source"] == source].copy()
    scenarios = sorted(sub["scenario"].unique())
    n = len(scenarios)
    if n == 0:
        print(f"No '{source}' rows in {title} -- skipping figure.")
        return
    ncols = 3
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), squeeze=False)

    for i, scen in enumerate(scenarios):
        ax = axes[i // ncols][i % ncols]
        scen_df = sub[sub["scenario"] == scen].sort_values("mean_importance")
        labels = [f"{r.kind}:{str(r.type)[:28]}" for r in scen_df.itertuples()]
        ax.barh(labels, scen_df["mean_importance"], color=PALETTE[0])
        ax.set_title(scen, fontsize=11, fontweight="bold")
        ax.set_xlim(0, 1)
        ax.set_xlabel("mean gnnexplainer importance")
        ax.tick_params(axis="y", labelsize=8)
        sns.despine(ax=ax)

    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    print(f"Saved figure to {FIG_DIR / filename}")
    plt.show()


plot_top_importance_grid(type_topn_normal_df, "Top-5 node/edge importance per scenario (normal)",
                          "fig_top_importance_normal.png")
plot_top_importance_grid(type_topn_ablation_df, "Top-5 node/edge importance per scenario (ablation)",
                          "fig_top_importance_ablation.png")

## Figure: anchor vs. other node types, normal vs. ablation

In [ ]:
def plot_anchor_comparison(anchor_df, title, filename):
    if anchor_df.empty:
        print(f"No rows for {title} -- skipping figure.")
        return
    melted = anchor_df.melt(id_vars=["scenario"],
                             value_vars=["anchor_mean_importance", "other_types_mean_importance"],
                             var_name="group", value_name="importance")
    melted["group"] = melted["group"].map({"anchor_mean_importance": "anchor",
                                            "other_types_mean_importance": "other types (avg)"})
    fig, ax = plt.subplots(figsize=(7, 0.55 * len(anchor_df) + 1.5))
    sns.barplot(data=melted, y="scenario", x="importance", hue="group", ax=ax, palette="muted")
    ax.set_xlim(0, 1)
    ax.set_xlabel("mean gnnexplainer importance")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(title=None, frameon=False, loc="lower right")
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    print(f"Saved figure to {FIG_DIR / filename}")
    plt.show()


plot_anchor_comparison(anchor_summary_normal_df, "Anchor vs. other node types -- normal", "fig_anchor_normal.png")
plot_anchor_comparison(anchor_summary_ablation_df, "Anchor vs. other node types -- ablation", "fig_anchor_ablation.png")

## v2 addition: feature-level explanation

Everything above explains importance at the **node** level ("was this
`building` node important"). This section adds a finer granularity:
**which NAMED FEATURE COMPONENT within that node** actually drove the
prediction -- e.g. was it a building's footprint `area`, its `height`,
or its `type_embed` (categorical building-type embedding) that mattered?

Mechanism (`explain.run_gnnexplainer_features`): the same GNNExplainer
optimization as above, but the learned mask has one value per **named
feature component per node** (see `explain.get_feature_components`)
instead of one value per node -- hooking each encoder's
`_assemble_raw_features` (the raw, pre-`input_proj` concatenated
feature vector) rather than `assemble_inputs`. Continuous fields
(position, area, height, ...) each get their own mask; a categorical
field's embedding block (`class_embed`, `type_embed`, `highway_embed`)
gets ONE shared mask value, since masking individual embedding
dimensions isn't interpretable -- only "was this categorical field
used at all" is.

**Cost note**: this runs a SECOND, independent GNNExplainer
optimization per point (feature-level masks are a different
parameterization from the node-level masks above, not derivable from
them) -- roughly doubles this notebook's total GNNExplainer time. It
reuses the exact same sampled points (same `EXPLAIN_SEED`,
`N_PER_CATEGORY`) as the node/edge-level pass above, so per-point
results are directly comparable across granularities.

**Edge features are NOT included here** -- true per-edge-attribute-
dimension masking (e.g. distinguishing `on_segment`'s distance
component from its highway-type component) would need an analogous
raw/projected split inside `assemble_edge_attrs`, scoped out for now;
edge importance stays at the existing per-edge granularity from the
node/edge-level section above.

### Run feature-level explanation -- normal and ablation, SEPARATELY (reuses the same sampled points)

In [ ]:
def run_feature_explanation_pass(scenarios, use_ablation):
    records = []
    label = "ablation" if use_ablation else "normal"
    for scenario in scenarios:
        print(f"\n=== [features/{label}] Scenario {scenario} ===")
        model, stats, best_repeat, tag = load_scenario_model(scenario, use_ablation=use_ablation)
        points = pick_points_to_explain(tag, best_repeat, n_per_category=N_PER_CATEGORY)
        print(f"  explaining {len(points)} points (feature-level)")

        for p in points:
            pid, cat = p["point_id"], p["category"]
            svg_raw = torch.load(SVG_DIR / f"{pid}.pt", weights_only=False)
            tvg_raw = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
            svg_norm, tvg_norm = ds.apply_normalization(copy.deepcopy(svg_raw), copy.deepcopy(tvg_raw), stats)

            try:
                recs = explain.explain_scenario_point_features(
                    model, scenario, svg_norm, tvg_norm, point_id=pid, category=cat,
                    gnnexplainer_epochs=GNNEXPLAINER_EPOCHS, gnnexplainer_lr=GNNEXPLAINER_LR)
                records.extend(recs)
            except Exception as e:
                print(f"    !! failed to explain (features) {pid} ({cat}): {type(e).__name__}: {e}")
    print(f"\n[features/{label}] total explained (point, branch) records: {len(records)}")
    return records


normal_feature_records = run_feature_explanation_pass(NORMAL_SCENARIOS, use_ablation=False)
ablation_feature_records = run_feature_explanation_pass(ABLATION_SCENARIOS, use_ablation=True)

### Aggregate into two tidy CSVs

In [ ]:
feature_df_normal = explain.aggregate_feature_explanations(normal_feature_records)
feature_df_ablation = explain.aggregate_feature_explanations(ablation_feature_records)

feature_df_normal.to_csv(EXPLAIN_DIR / "feature_explanations_capacity_revision_normal.csv", index=False)
feature_df_ablation.to_csv(EXPLAIN_DIR / "feature_explanations_capacity_revision_ablation.csv", index=False)

print(f"Saved {len(feature_df_normal)} rows to {EXPLAIN_DIR / 'feature_explanations_capacity_revision_normal.csv'}")
print(f"Saved {len(feature_df_ablation)} rows to {EXPLAIN_DIR / 'feature_explanations_capacity_revision_ablation.csv'}")
display(feature_df_normal.head(10))
display(feature_df_ablation.head(10))

### Per-scheme feature-component importance summary

In [ ]:
feature_pivot_normal_df, feature_topn_normal_df = explain.build_feature_importance_report(feature_df_normal, top_n=5)
feature_pivot_ablation_df, feature_topn_ablation_df = explain.build_feature_importance_report(feature_df_ablation, top_n=5)

feature_pivot_normal_df.to_csv(EXPLAIN_DIR / "feature_importance_summary_normal.csv", index=False)
feature_topn_normal_df.to_csv(EXPLAIN_DIR / "feature_importance_top5_normal.csv", index=False)
feature_pivot_ablation_df.to_csv(EXPLAIN_DIR / "feature_importance_summary_ablation.csv", index=False)
feature_topn_ablation_df.to_csv(EXPLAIN_DIR / "feature_importance_top5_ablation.csv", index=False)

print(f"[normal]   saved {len(feature_pivot_normal_df)} summary rows, {len(feature_topn_normal_df)} top-5 rows")
print(f"[ablation] saved {len(feature_pivot_ablation_df)} summary rows, {len(feature_topn_ablation_df)} top-5 rows")
display(feature_topn_normal_df)
display(feature_topn_ablation_df)

## Figure: top-5 feature-component importance per scenario, seaborn, L-shaped axes

Same convention as the node/edge-level figures above -- `sns.despine()`
(L-shaped, only left+bottom spines), muted palette. Each bar is one
`node_type.feature` pair (e.g. `building.height`, `signage.class_embed`),
not a whole node type.

In [ ]:
def plot_top_feature_importance_grid(topn_df, title, filename):
    scenarios = sorted(topn_df["scenario"].unique())
    n = len(scenarios)
    if n == 0:
        print(f"No rows in {title} -- skipping figure.")
        return
    ncols = 3
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), squeeze=False)

    for i, scen in enumerate(scenarios):
        ax = axes[i // ncols][i % ncols]
        scen_df = topn_df[topn_df["scenario"] == scen].sort_values("mean_importance")
        labels = [f"{r.node_type}.{r.feature}" for r in scen_df.itertuples()]
        ax.barh(labels, scen_df["mean_importance"], color=PALETTE[1])
        ax.set_title(scen, fontsize=11, fontweight="bold")
        ax.set_xlim(0, 1)
        ax.set_xlabel("mean gnnexplainer importance (feature-level)")
        ax.tick_params(axis="y", labelsize=8)
        sns.despine(ax=ax)

    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    print(f"Saved figure to {FIG_DIR / filename}")
    plt.show()


plot_top_feature_importance_grid(feature_topn_normal_df, "Top-5 feature-component importance per scenario (normal)",
                                  "fig_top_feature_importance_normal.png")
plot_top_feature_importance_grid(feature_topn_ablation_df, "Top-5 feature-component importance per scenario (ablation)",
                                  "fig_top_feature_importance_ablation.png")

In [ ]:
print("08g_v2 feature-level explainability run complete.")
print(f"[normal]   {feature_df_normal['point_id'].nunique()} unique points, "
      f"{feature_df_normal['scenario'].nunique()} scenario/branch tags, "
      f"{feature_df_normal['feature'].nunique()} distinct feature components explained.")
print(f"[ablation] {feature_df_ablation['point_id'].nunique()} unique points, "
      f"{feature_df_ablation['scenario'].nunique()} scenario/branch tags.")
print()
print("New (v2) outputs, normal:   feature_explanations_capacity_revision_normal.csv,")
print("                            feature_importance_summary_normal.csv, feature_importance_top5_normal.csv")
print("New (v2) outputs, ablation: feature_explanations_capacity_revision_ablation.csv,")
print("                            feature_importance_summary_ablation.csv, feature_importance_top5_ablation.csv")
print("(node/edge-level outputs from the section above are unchanged from 08g)")